In [3]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/tw_rp_main/jupyter-notebooks/training_sets/train_20251002_043005.csv"  # queries
B_PATH = "/home/ubuntu/tw_rp_main/datasets/500_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/r5_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/r5_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/tw_rp_main/similarity_scores/cross_similar_posts_k3_500_r5.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['selftext'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id   subreddit                                              title  \
 0   on02g1    Pregnant  Gender disappointment somewhat. FTM with a BOY...   
 1  18p5rk7   BabyBumps                      Anxiety- miscarriage concerns   
 2   wgoybd    abortion     I regret my abortion, but I remain pro-choice.   
 3  1l494t2  depression                                 i hate being trans   
 4   oj3t14     assault                 looking for help years after abuse   
 
                                             selftext          created_utc  \
 0  I know this is a thing, and quite common, but ...  2021-07-18 21:41:32   
 1  I run anxious but I’ve been 400% more anxious ...  2023-12-23 13:32:35   
 2  Maybe it was because my decision was so rushed...   2022-08-05 7:25:39   
 3  it's so hard because you know that no one will...  2025-06-05 20:05:23   
 4  lately i’ve been feeling trapped by the abuse ...   2021-07-13 0:01:48   
 
                                                  url  Tags 

In [5]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
r5_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r5_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
r5_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r5_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [6]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [8]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(r5_emb_A, r5_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/tw_rp_main/similarity_scores/cross_similar_posts_k3_500_r5.json
